In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Delta_Project").getOrCreate()
data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),  # updated record
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),  # invalid amount
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)   # duplicate
]

columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]
df = spark.createDataFrame(data, columns)
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,null,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,null,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
# Create the schemas in your current catalog
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

DataFrame[]

In [0]:
# Save raw data to Bronze schema
df.write.mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("bronze.orders_raw")

print("Bronze data stored in 'bronze' schema.")

Bronze data stored in 'bronze' schema.


In [0]:
from pyspark.sql.functions import col, avg, when
sliver_df=df.withColumn("amount",col("amount").cast("float"))\
    .withColumn("quantity",col("quantity").cast("int"))\
     .withColumn("date",col("date").cast("date"))


sliver_df.show()

+--------+-----------+----------+-----------+---------+----------+--------+--------+
|order_id|customer_id|   product|   category|     city|      date|  amount|quantity|
+--------+-----------+----------+-----------+---------+----------+--------+--------+
|     101|       C001|    Laptop|Electronics|Hyderabad|2024-01-01| 50000.0|       1|
|     102|       C002|    Mobile|Electronics|  Chennai|2024-01-02|    NULL|       2|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-03| 20000.0|       1|
|     104|       C003|    Laptop|Electronics|    Delhi|2024-01-04| 55000.0|       1|
|     105|       C002|    Mobile|Electronics|  Chennai|2024-01-05| 18000.0|       1|
|     106|       C004|     Watch|Accessories|   Mumbai|2024-01-06|  8000.0|       1|
|     103|       C001|    Tablet|Electronics|Hyderabad|2024-01-07| 22000.0|       1|
|     107|       C005|Headphones|Accessories|     NULL|2024-01-08|  3000.0|       1|
|     108|       C006|    Laptop|Electronics|Bangalore|2024-01-09

In [0]:
from pyspark.sql.functions import col, abs


sliver_df = sliver_df.withColumn("amount", abs(col("amount")))

sliver_df = sliver_df.fillna({"amount": 0, "city": "Unknown"})


sliver_df = sliver_df.dropDuplicates()

sliver_df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000.0,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,0.0,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000.0,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000.0,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000.0,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000.0,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000.0,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000.0,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,45000.0,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000.0,2


In [0]:
from pyspark.sql.functions import col, abs, row_number
from pyspark.sql.window import Window
window_spec=Window.partitionBy("order_id").orderBy(col("date").desc())
sliver_df=sliver_df.withColumn("rn",row_number().over(window_spec))\
            .filter(col("rn")==1)\
             .drop("rn")

sliver_df.sort("order_id").display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000.0,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,0.0,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000.0,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000.0,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000.0,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000.0,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000.0,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,45000.0,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000.0,2


In [0]:
# Save cleaned data to Silver schema
sliver_df.write.mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("silver.orders_cleaned")

print("Silver data stored in 'silver' schema.")

Silver data stored in 'silver' schema.


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

DataFrame[]

# **Tasks:**

# **Aggregations**

In [0]:
from pyspark.sql.functions import sum, count, desc

# Load data from the Silver Layer
silver_data = spark.table("silver.orders_cleaned")


# 1. Calculate total sales per product from your Silver data
product_sales=sliver_df.groupBy("product")\
    .agg(sum("amount").alias("total_product_sales"))\
    .orderBy(desc("total_product_sales"))


# 2. Display the result
product_sales.display()
# 5. Store it in the Gold Layer
# Using 'gold.total_sales_by_product'
product_sales.write.mode("overwrite") \
    .saveAsTable("gold.total_sales_by_product")

print("Gold table 'total_sales_by_product' created successfully!")



product,total_product_sales
Laptop,150000.0
Mobile,33000.0
Tablet,22000.0
Watch,8000.0
Headphones,3000.0


Gold table 'total_sales_by_product' created successfully!


In [0]:
#Total sales by category
category_sales=sliver_df.groupBy("category")\
    .agg(sum("amount").alias("total_category_sales"))\
    .orderBy(desc("total_category_sales"))



category_sales.display()
# 5. Store it in the Gold Layer
# Using 'gold.total_sales_by_category'
category_sales.write.mode("overwrite") \
    .saveAsTable("gold.total_sales_by_category")

print("Gold table 'total_sales_by_category' created successfully!")


category,total_category_sales
Electronics,205000.0
Accessories,11000.0


Gold table 'total_sales_by_category' created successfully!


In [0]:
#Total sales by city
city_sales=sliver_df.groupBy("city")\
    .agg(sum("amount").alias("total_city_sales"))\
    .orderBy(desc("total_city_sales"))



city_sales.display()
# 5. Store it in the Gold Layer
# Using 'gold.total_sales_by_city'
city_sales.write.mode("overwrite") \
    .saveAsTable("gold.total_sales_by_city")

print("Gold table 'total_sales_by_city' created successfully!")

city,total_city_sales
Hyderabad,72000.0
Delhi,55000.0
Bangalore,45000.0
Chennai,33000.0
Mumbai,8000.0
Unknown,3000.0


Gold table 'total_sales_by_city' created successfully!


# **Customer Metrics**

In [0]:
#Total orders per customer
total_orders=sliver_df.groupBy("customer_id")\
    .agg(count("order_id").alias("total_orders"))\
    .orderBy(desc("total_orders"))




total_orders.display()
# 5. Store it in the Gold Layer
# Using 'gold.total_orders_by_customer'
total_orders.write.mode("overwrite") \
    .saveAsTable("gold.total_orders_by_customer")

print("Gold table 'total_orders_by_customer' created successfully!")

customer_id,total_orders
C001,2
C002,2
C003,1
C004,1
C005,1
C006,1
C007,1


Gold table 'total_orders_by_customer' created successfully!


# **Top Analysis**

In [0]:
# Top selling product
# Corrected the column name from 'total_product-sales' to 'total_product_sales'
top_selling_product = product_sales.orderBy(desc("total_product_sales")).limit(1)
top_selling_product.display()

product,total_product_sales
Laptop,150000.0


In [0]:
#Top customer
top_customer=total_orders.orderBy(desc("total_orders")).limit(1)
top_customer.display()
# 5. Store it in the Gold Layer
# Using 'gold.top_customer'
top_customer.write.mode("overwrite") \
    .saveAsTable("gold.top_customer")

print("Gold table 'top_customer' created successfully!")

customer_id,total_orders
C001,2


Gold table 'top_customer' created successfully!
